In [ ]:
# 构造市值加权收益、 2× 3 组合和 CH3 / FF3 / CH4 / FF5 因子。
def get_vw_return(data):
    # 组合市值加权收益；对应 portfolio_5.get_groups_ch3 里的 vw_ret，另加正市值/有限值防御。
    has_positive_market_value = data[monthly_market_value].gt(0)
    has_finite_market_value = np.isfinite(data[monthly_market_value])
    has_finite_return = np.isfinite(data[monthly_excess_ret])
    valid = (
        has_positive_market_value
        & has_finite_market_value
        & has_finite_return
    )
    data = data.loc[valid]
    if data.empty:
        return np.nan
    total_market_value = data[monthly_market_value].sum()
    weights = data[monthly_market_value] / total_market_value
    return (data[monthly_excess_ret] * weights).sum()

def get_size_value_ports(monthly_final, signal):
    # 独立 2×3 排序：市值中位数分 s/b、信号 30/70 分 l/m/h；六组合市值加权后
    # size=(s−b)/3、value=(h−l)/2，与 portfolio_5.get_groups_ch3 的 SMB/VMG 公式一致。
    data = monthly_final.dropna(
        subset=[monthly_market_value]
    ).copy()
    signal_count = data.groupby(ym)[signal].transform('count')
    data = data.loc[signal_count.ge(4)]
    data['size_group'] = data.groupby(ym)[monthly_market_value].transform(
        lambda values: pd.qcut(
            values.rank(method='first'),
            q=[0, 0.5, 1],
            labels=['s', 'b'],
        )
    ).astype(str)
    data = data.dropna(subset=[signal])
    data['signal_group'] = data.groupby(ym)[signal].transform(
        lambda values: pd.qcut(
            values.rank(method='first'),
            q=[0, 0.3, 0.7, 1],
            labels=['l', 'm', 'h'],
        )
    ).astype(str)
    data['portfolio_group'] = (
        data['size_group'] + '/' + data['signal_group']
    )
    portfolio_columns = [monthly_market_value, monthly_excess_ret]
    ports = data.groupby([ym, 'portfolio_group'])[portfolio_columns]
    ports = ports.apply(get_vw_return).unstack()
    portfolio_names = ['s/l', 's/m', 's/h', 'b/l', 'b/m', 'b/h']
    ports = ports.reindex(columns=portfolio_names)
    size = (
        ports['s/l'] + ports['s/m'] + ports['s/h']
        - ports['b/l'] - ports['b/m'] - ports['b/h']
    ) / 3
    value = (
        ports['s/h'] + ports['b/h'] - ports['s/l'] - ports['b/l']
    ) / 2
    return ports, size, value

def get_ch3_factors(monthly_all, cut=0.30):
    # CH-3 中国三因子；与 portfolio_5.get_groups_ch3 同流程（剔壳 30% → 2×3 →
    # 市值加权 → SMB/VMG/MKT），剔壳断点按传入的形成股票池（monthly_all）计算。
    data = monthly_all.copy()
    size_pct = data.groupby(ym)[monthly_market_value].rank(
        pct=True, method='average'
    )
    data = data[size_pct.gt(cut)]
    ports, size, value = get_size_value_ports(data, monthly_ep)
    market_columns = [monthly_market_value, monthly_excess_ret]
    market = data.groupby(ym)[market_columns].apply(get_vw_return)
    return pd.DataFrame(
        {
            mkt: market,
            smb: size,
            vmg: value,
        }
    )


# FF-3 / CH-4 / FF-5 复用同一套 2×3 市值加权排序；portfolio_5 只对 CH-3 做了专用实现。


In [ ]:
f_ch3 = get_ch3_factors(monthly_all, cut=0.30)

ff_ports, ff_size, ff_value = get_size_value_ports(
    monthly_final, monthly_bm
)

f_ff3 = pd.DataFrame(
    {
        mkt: f_ch3[mkt],
        'ff_size_factor': ff_size,
        'ff_value_factor': ff_value,
    }
)

turn_ports, turn_size, optimism = get_size_value_ports(
    monthly_final, monthly_abn_turnover_1m
)

f_ch4 = pd.DataFrame(
    {
        mkt: f_ch3[mkt],
        smb: (f_ch3[smb] + turn_size) / 2,
        vmg: f_ch3[vmg],
        'turnover_factor_pmo': -optimism,
    }
)

profit_ports, profit_size, profit_value = get_size_value_ports(
    monthly_final, monthly_roe
)

invest_ports, invest_size, invest_growth = get_size_value_ports(
    monthly_final, monthly_asset_growth
)

f_ff5 = pd.DataFrame(
    {
        mkt: f_ch3[mkt],
        'ff_size_factor': (
            ff_size + profit_size + invest_size
        ) / 3,
        'ff_value_factor': ff_value,
        'ff_profit_factor': profit_value,
        'ff_invest_factor': -invest_growth,
    }
)

print('查看因子矩阵 F 的 shape 与完整月份')

factor_sets = [
    ('CH-3', f_ch3),
    ('FF-3', f_ff3),
    ('CH-4', f_ch4),
    ('FF-5 proxy', f_ff5),
]

for name, factors in factor_sets:
    print(name, factors.shape, factors.dropna().shape)

display(f_ch3.head())


# 再构造十项已报道异象的单变量与市值中性多空组合。
# 对应 portfolio_5 的 get_groups_uncon / get_groups_neutral；方向声明见 anomaly_specs。

